In [1]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table

ModuleNotFoundError: No module named 'utils'

In [ ]:
df = pd.read_csv("../../data/Total Graduates by Age Group.csv")
df.head(10)

In [ ]:
def transform_grad_by_age(file_path):
    df = pd.read_csv(file_path)

    # 1. Pivot dataframe
    df_long = df.melt(
        id_vars=["statistics"],
        var_name="year",
        value_name="value"
    )

    # 2. Parse statistics into age_group and qualification
    def parse_stats(stat_name):
        if "_" in stat_name:
            age_part, qual_part = stat_name.split("_", 1)
            return age_part.strip(), qual_part.strip()
        return "Total", stat_name.strip()

    df_long[['age_group', 'qual_cat']] = df_long['statistics'].apply(
        lambda x: pd.Series(parse_stats(x))
    )

    # 3. Extract qualification from qual_cat
    def extract_qualification(qual_cat):
        if pd.isna(qual_cat):
            return None
        if "_" in qual_cat:
            return qual_cat.split("_", 1)[1]
        return qual_cat

    df_long['qualification'] = df_long['qual_cat'].apply(extract_qualification)

    # 4. Convert numeric and scale
    df_long['total_graduate'] = pd.to_numeric(df_long['value'], errors='coerce')
    df_long = df_long.dropna(subset=['total_graduate'])
    df_long['total_graduate'] = df_long['total_graduate'] * 1000

    df_final = df_long[['year', 'age_group', 'qualification', 'total_graduate']].copy()
    df_final = df_final.sort_values(['year', 'age_group', 'qualification']).reset_index(drop=True)

    return df_final

In [ ]:
df_final = transform_grad_by_age("../../data/Total Graduates by Age Group.csv")
df_final.head(10)

In [ ]:
write_table(df_final, "sc_bronze", "dosm_graduates_age")